In [1]:
# todo: add y label in build_hetero_graph_cancer method
# todo: change path of saved graphs
# todo: make homogenous graph 

In [2]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool

class GraphLevelClassifier(nn.Module):
    def __init__(self, in_channels=320, hidden_dim=128):  # Fixed: use explicit input dimension
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),  # Added dropout for regularization
            nn.Linear(64, 1)  # Binary classification
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        pooled = global_mean_pool(x, batch)
        return self.classifier(pooled).view(-1)


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from torch_geometric.data import Dataset
import torch
import os

class PatientGraphDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()
        self.root_dir = root_dir
        self.files = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith('.pt')]
        self.transform = transform

    def len(self):
        return len(self.files)

    def get(self, idx):
        self.data, self.df = torch.load(self.files[idx], weights_only=False)
        if self.transform:
            self.data = self.transform(self.data)
        return self.data


In [4]:
from sklearn.metrics import accuracy_score, roc_auc_score
import torch

def train_graph_classifier(model, train_loader, test_loader, epochs=20, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        all_preds, all_labels = [], []

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = loss_fn(logits, batch.y.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())

        train_acc = accuracy_score(all_labels, [p > 0.5 for p in all_preds])
        train_auc = roc_auc_score(all_labels, all_preds)
        avg_loss = total_loss / len(train_loader)

        # Evaluation
        model.eval()
        test_preds, test_labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                logits = model(batch)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())
                test_labels.extend(batch.y.cpu().numpy())
        
        test_acc = accuracy_score(test_labels, [p > 0.5 for p in test_preds])
        test_auc = roc_auc_score(test_labels, test_preds)

        print(f"[Epoch {epoch}] Loss: {avg_loss:.4f} | Train Acc: {train_acc:.3f} | Train AUC: {train_auc:.3f} | Test Acc: {test_acc:.3f} | Test AUC: {test_auc:.3f}")

In [14]:
full_dataset = PatientGraphDataset("/dsi/sbm/chen2/graph_data/0.05_False_True")


# Combine and shuffle
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

Looking for files in: /dsi/sbm/chen2/graph_data/0.05_False_True/
Directory exists: True
Found 98 files
First 10 files: ['healthy_38.clones_TRB.tsv.pt', 'sick_S35.clones_TRB.tsv.pt', 'healthy_27.clones_TRB.tsv.pt', 'sick_49.clones_TRB.tsv.pt', 'healthy_34.clones_TRB.tsv.pt', 'sick_3.clones_TRB.tsv.pt', 'healthy_25.clones_TRB.tsv.pt', 'healthy_18.clones_TRB.tsv.pt', 'sick_S29.clones_TRB.tsv.pt', 'sick_S33.clones_TRB.tsv.pt']

Loading sample file: healthy_38.clones_TRB.tsv.pt
Graph data shape: torch.Size([11827, 320])
Edge index shape: torch.Size([2, 92904])
Has labels: True
Labels: 0.0
DataFrame shape: (12172, 19)
DataFrame columns: ['cloneId', 'readCount', 'readFraction', 'targetSequences', 'targetQualities', 'allVHitsWithScore', 'allDHitsWithScore', 'allJHitsWithScore', 'allCHitsWithScore', 'allVAlignments', 'allDAlignments', 'allJAlignments', 'allCAlignments', 'nSeqCDR3', 'minQualCDR3', 'aaSeqCDR3', 'refPoints', 'cdr3_truncated', 'tcr_index']
DataFrame sample:
   cloneId  readCount  r

In [6]:
# Get metadata from one example

model = GraphLevelClassifier()
train_graph_classifier(model, train_loader, test_loader, epochs=30)

Epoch 1/30
[Epoch 1] Loss: 0.6980 | Train Acc: 0.462 | Train AUC: 0.312 | Test Acc: 0.400 | Test AUC: 0.188
Epoch 2/30
[Epoch 2] Loss: 0.6949 | Train Acc: 0.372 | Train AUC: 0.387 | Test Acc: 0.600 | Test AUC: 0.786
Epoch 3/30
[Epoch 3] Loss: 0.6945 | Train Acc: 0.372 | Train AUC: 0.405 | Test Acc: 0.600 | Test AUC: 0.792
Epoch 4/30
[Epoch 4] Loss: 0.6942 | Train Acc: 0.436 | Train AUC: 0.396 | Test Acc: 0.600 | Test AUC: 0.792
Epoch 5/30


KeyboardInterrupt: 